In [1]:
import deltafq as dfq # repo: https://github.com/Delta-F/deltafq
import pandas as pd

In [2]:
# Fetch market data
fetcher = dfq.data.DataFetcher()
fetcher.initialize()
data = fetcher.fetch_stock_data('AAPL', '2023-01-01', '2023-12-31')

# Clean and validate data
cleaner = dfq.data.DataCleaner()
cleaner.initialize()
cleaned_data = cleaner.clean_price_data(data)

validator = dfq.data.DataValidator()
validator.initialize()
validator.validate_price_data(cleaned_data)

# Create and test a strategy
class SimpleMAStrategy(dfq.strategy.BaseStrategy):
    def __init__(self, fast_period=10, slow_period=20):
        super().__init__()
        self.fast_period = fast_period
        self.slow_period = slow_period
    
    def generate_signals(self, data):
        fast_ma = data['close'].rolling(window=self.fast_period).mean()
        slow_ma = data['close'].rolling(window=self.slow_period).mean()
        import numpy as np
        signals = np.where(fast_ma > slow_ma, 1, np.where(fast_ma < slow_ma, -1, 0))
        return pd.Series(signals, index=data.index)

strategy = SimpleMAStrategy()
strategy.initialize()
results = strategy.run(cleaned_data)

# Run backtest
engine = dfq.backtest.BacktestEngine(initial_capital=100000)
engine.initialize()
backtest_results = engine.run_backtest(strategy, cleaned_data)

# Run paper trading
simulator = dfq.trading.PaperTradingSimulator(initial_capital=100000)
simulator.initialize()
portfolio_summary = simulator.run_strategy(strategy, cleaned_data)

2025-10-13 15:27:32,278 - DataFetcher - INFO - Initializing data fetcher with source: yahoo
2025-10-13 15:27:32,280 - DataFetcher - INFO - Fetching data for AAPL from 2023-01-01 to 2023-12-31
2025-10-13 15:27:32,282 - DataCleaner - INFO - Initializing data cleaner
2025-10-13 15:27:32,285 - DataCleaner - INFO - Cleaned data: 365 -> 365 rows
2025-10-13 15:27:32,286 - DataValidator - INFO - Initializing data validator
2025-10-13 15:27:32,287 - DataValidator - INFO - Price data validation passed
2025-10-13 15:27:32,288 - SimpleMAStrategy - INFO - Initializing strategy: SimpleMAStrategy
2025-10-13 15:27:32,288 - SimpleMAStrategy - INFO - Running strategy: SimpleMAStrategy
2025-10-13 15:27:32,289 - BacktestEngine - INFO - Initializing backtest engine with capital: 100000
2025-10-13 15:27:32,290 - BacktestEngine - INFO - Starting backtest
2025-10-13 15:27:32,391 - BacktestEngine - INFO - Backtest completed. Total return: 0.00%
2025-10-13 15:27:32,391 - PaperTradingSimulator - INFO - Initializ